# Module 2.6 — Regular Expressions & Text Processing

### Python Mastery · Track 2: Data Structures & Iteration

A regular expression is a compact language for describing patterns in text. With the `re` module you can search, extract, validate, and transform strings that would be awkward to handle with ordinary string methods. This module covers the core functions, groups, flags, the greedy-versus-lazy distinction, and when not to reach for a regular expression.

**How to use this notebook**

- Read each explanation, then run the code cell beneath it.
- Adjust the patterns and test strings and observe the matches.
- Attempt the exercises before consulting the solutions.

### Learning objectives

After completing this module you will be able to:

1. Use `match`, `search`, `findall`, `finditer`, `sub`, and `split`.
2. Read common pattern syntax: character classes, quantifiers, and anchors.
3. Capture parts of a match with groups and named groups.
4. Apply flags and understand greedy versus lazy matching.
5. Recognise when a regular expression is the wrong tool.

**Prerequisites:** Track 1 (especially Module 1.7) and Modules 2.1 to 2.5.

---

## Part 1 · Patterns, Raw Strings, and the Core Functions

A pattern is written as a string. Because regular expressions use the backslash heavily, patterns are written as **raw strings** with an `r` prefix (for example `r"\d+"`), which stops Python from interpreting the backslashes itself.

A few building blocks to start with:

- `\d` a digit, `\w` a word character (letter, digit, or underscore), `\s` whitespace.
- `+` one or more, `*` zero or more, `?` zero or one, `{n}` exactly n.
- `.` any character; `^` start of the string; `$` end of the string.

The main functions are `re.search` (find the first match anywhere), `re.match` (match only at the start), `re.findall` (all matches as a list), `re.finditer` (all matches as objects), `re.sub` (replace), and `re.split` (split on a pattern).

In [ ]:
import re

text = "Order 1234 shipped on 2025-05-30, order 5678 pending."

# search finds the first match anywhere and returns a match object (or None).
m = re.search(r"\d+", text)
print("first number:", m.group(), "at", m.span())

# match only succeeds at the very start of the string.
print("match at start:", re.match(r"\d+", text))      # None: the text starts with 'Order'
print("match at start:", re.match(r"Order", text))     # succeeds

# findall returns every match as a list of strings.
print("all numbers:", re.findall(r"\d+", text))

A match object carries useful information: `group()` is the matched text, `start()` and `end()` are its position, and `span()` is the pair. Always check for `None` before using a match, since a failed search returns `None`.

In [ ]:
import re

m = re.search(r"\d{4}-\d{2}-\d{2}", text)   # a date-like pattern: four-two-two digits
if m:
    print("found date:", m.group(), "between", m.start(), "and", m.end())
else:
    print("no date found")

## Part 2 · Character Classes and Quantifiers

Square brackets define a **character class**, matching any one character inside. A range uses a hyphen, and a leading `^` negates the class. Quantifiers control how many times the preceding piece repeats.

In [ ]:
import re

sample = "Cat cot CUT cit c@t"

print("c, any vowel, t:", re.findall(r"c[aeiou]t", sample))        # cat cot cit
print("c, NOT a letter, t:", re.findall(r"c[^a-zA-Z]t", sample))   # c@t

phones = "Call 080-1234 or 0888-99 today"
print("digit groups of 2 to 4:", re.findall(r"\d{2,4}", phones))

words = "a bb ccc dddd"
print("runs of 2+ letters:", re.findall(r"[a-z]{2,}", words))

## Part 3 · Groups, Named Groups, and Backreferences

Parentheses create a **group**, which captures part of a match for later use. `group(1)` returns the first captured group. Naming a group with `(?P<name>...)` lets you retrieve it by name, which is clearer. A **backreference** matches the same text a group already captured, which is useful for finding repeats.

In [ ]:
import re

line = "2025-05-30"

# Three capture groups for year, month, day.
m = re.search(r"(\d{4})-(\d{2})-(\d{2})", line)
print("whole match:", m.group(0))
print("year:", m.group(1), "month:", m.group(2), "day:", m.group(3))
print("all groups:", m.groups())

# Named groups read far better than numbered ones.
m2 = re.search(r"(?P<year>\d{4})-(?P<month>\d{2})-(?P<day>\d{2})", line)
print("named year:", m2.group("year"), "named day:", m2.group("day"))
print("as a dict:", m2.groupdict())

# A backreference finds a doubled word. \1 means 'the same text group 1 matched'.
doubled = "the the cat sat on on the mat"
print("doubled words:", re.findall(r"\b(\w+)\s+\1\b", doubled))

## Part 4 · Substitution, Splitting, Flags, and Compiling

`re.sub(pattern, replacement, text)` replaces every match. The replacement can reference captured groups with `\1` or `\g<name>`. `re.split` splits a string wherever the pattern occurs. **Flags** modify behaviour, the most common being `re.IGNORECASE`. When a pattern is reused, `re.compile` builds it once for efficiency and tidier code.

In [ ]:
import re

# Substitution: mask all numbers.
text = "Cards 1234 5678 and 9999 0000 were used."
print("masked:", re.sub(r"\d", "*", text))

# Substitution reordering with group references: swap 'first last' to 'last, first'.
print(re.sub(r"(\w+) (\w+)", r"\2, \1", "Ada Lovelace"))

# Splitting on any run of non-word characters.
messy = "apple, banana;  cherry|date"
print("split:", re.split(r"[,;|]\s*", messy))

# Flags: case-insensitive search.
print("ignorecase:", re.findall(r"python", "Python PYTHON python", re.IGNORECASE))

# Compiling a reused pattern.
word_pattern = re.compile(r"\b\w{4,}\b")   # words of four or more letters
print("long words:", word_pattern.findall("the quick brown fox jumps"))

## Part 5 · Greedy Versus Lazy, and When Not to Use Regex

By default quantifiers are **greedy**: they match as much as possible. Adding `?` makes them **lazy**, matching as little as possible. This matters when a pattern could match different amounts of text.

Regular expressions are powerful but not always appropriate. For fixed substrings prefer `in` or `str.find`; for simple splitting prefer `str.split`. For deeply nested or structured formats such as HTML, JSON, or XML, use a real parser instead, because such formats are beyond what regular expressions can reliably handle.

In [ ]:
import re

html = "<b>bold</b> and <i>italic</i>"

# Greedy: <.+> grabs from the first < to the LAST >, swallowing everything between.
print("greedy:", re.findall(r"<.+>", html))

# Lazy: <.+?> stops at the first >, matching each tag individually.
print("lazy:", re.findall(r"<.+?>", html))

# A case where a plain string method is the better choice:
sentence = "the password is secret123"
print("substring check (no regex needed):", "password" in sentence)

---

## Worked Examples

| Examples | Level |
|---|---|
| 1 and 2 | Easy |
| 3 and 4 | Medium |
| 5 and 6 | Difficult |

### Example 1 — Extracting all numbers (Easy)

In [ ]:
import re
text = "Room 12 holds 30 chairs and 4 tables."
print(re.findall(r"\d+", text))     # ['12', '30', '4']
# Experiment: change \d+ to \d to match single digits instead of whole numbers.

### Example 2 — Checking a simple format (Easy)

In [ ]:
import re

def looks_like_pincode(s):
    # ^ and $ anchor the whole string; exactly six digits and nothing else.
    return re.match(r"^\d{6}$", s) is not None

for code_str in ["411001", "41100", "4110AB"]:
    print(code_str, "->", looks_like_pincode(code_str))

### Example 3 — Parsing dates into parts (Medium)

In [ ]:
import re

log = "events: 2025-01-15, 2025-12-03, and 2024-06-21 recorded"
pattern = re.compile(r"(?P<y>\d{4})-(?P<m>\d{2})-(?P<d>\d{2})")

for match in pattern.finditer(log):
    print(f"year={match.group('y')} month={match.group('m')} day={match.group('d')}")

### Example 4 — Cleaning and normalising text (Medium)

In [ ]:
import re

raw = "Hello,,,   world!!!   This---is    messy."

# Collapse any run of whitespace to a single space.
step1 = re.sub(r"\s+", " ", raw)
# Reduce repeated punctuation to a single mark.
step2 = re.sub(r"([,.!?-])\1+", r"\1", step1)
print(step2.strip())

### Example 5 — Extracting key-value pairs (Difficult)

In [ ]:
import re

config = "host=localhost; port=8080; debug=true; name=my app"

# Capture a key (word characters) and a value (anything up to ; or end).
pattern = re.compile(r"(\w+)\s*=\s*([^;]+)")
settings = {}
for key, value in pattern.findall(config):
    settings[key] = value.strip()

print(settings)
print("port as int:", int(settings["port"]))

### Example 6 — A small tokeniser with alternation (Difficult)

In [ ]:
import re

expression = "12 + 34 * 5 - 6"

# Alternation (|) tries each option; here a number OR one of the operators.
token_pattern = re.compile(r"\d+|[+\-*/]")
tokens = token_pattern.findall(expression)
print("tokens:", tokens)

numbers = [int(t) for t in tokens if t.isdigit()]
operators = [t for t in tokens if not t.isdigit()]
print("numbers:", numbers, "| operators:", operators)

---

## Exercises

**Exercise 1 (Easy).** Using `findall`, extract every word (run of letters) from the sentence below.

In [ ]:
import re
sentence = "Hello, world! Regex is powerful."
# Your solution here


**Exercise 2 (Easy).** Write a function that returns `True` if a string consists only of lowercase letters and is between 3 and 8 characters long, using an anchored pattern.

In [ ]:
import re
# Your solution here


**Exercise 3 (Medium).** From the text below, extract all the prices (a currency symbol is not present; they look like one or more digits, optionally followed by a dot and two digits). Return them as a list of strings.

In [ ]:
import re
text = "Items cost 40, 19.99, 200 and 5.50 today."
# Your solution here


**Exercise 4 (Medium).** Using `re.sub`, replace every sequence of one or more spaces in the string below with a single hyphen, to make a simple slug.

In [ ]:
import re
title = "  My   First    Blog Post  "
# Your solution here (remember to strip the ends first)


**Exercise 5 (Difficult).** Using named groups, parse each line below of the form `name: score` into a dictionary mapping name to integer score.

In [ ]:
import re
lines = ["asha: 88", "ben: 92", "chen: 75"]
# Your solution here


---

## Solutions

**Solution 1**

In [ ]:
import re
sentence = "Hello, world! Regex is powerful."
print(re.findall(r"[A-Za-z]+", sentence))

**Solution 2**

In [ ]:
import re
def valid(s):
    return re.match(r"^[a-z]{3,8}$", s) is not None

for test in ["cat", "ab", "lowercase", "Mixed"]:
    print(test, "->", valid(test))

**Solution 3**

In [ ]:
import re
text = "Items cost 40, 19.99, 200 and 5.50 today."
# \d+ then an optional group of a dot and exactly two digits.
print(re.findall(r"\d+(?:\.\d{2})?", text))

**Solution 4**

In [ ]:
import re
title = "  My   First    Blog Post  "
slug = re.sub(r"\s+", "-", title.strip())
print(slug)

**Solution 5**

In [ ]:
import re
lines = ["asha: 88", "ben: 92", "chen: 75"]
pattern = re.compile(r"(?P<name>\w+):\s*(?P<score>\d+)")
result = {}
for line in lines:
    m = pattern.match(line)
    result[m.group("name")] = int(m.group("score"))
print(result)

---

## Summary and Key Points

- Write patterns as raw strings (`r"..."`). Core functions are `search` (first match anywhere), `match` (start only), `findall` (list of strings), `finditer` (match objects), `sub` (replace), and `split`.
- Character classes `[...]` match one of a set; `\d`, `\w`, `\s` are common shorthands; quantifiers `+ * ? {n,m}` control repetition; `^` and `$` anchor to start and end.
- Parentheses capture groups, retrievable by number or, more clearly, by name with `(?P<name>...)`; backreferences match previously captured text.
- `re.sub` can reference groups in its replacement, flags such as `re.IGNORECASE` modify matching, and `re.compile` builds a reusable pattern.
- Quantifiers are greedy by default and lazy with a trailing `?`. Use plain string methods for simple cases and a real parser for structured formats rather than regular expressions.

### Next module: 2.7 — Dates, Times & Durations

The next module covers working with calendar dates and clock times, computing durations, formatting and parsing, and handling time zones correctly.